# Data Cleaning 

## Project Overview

This notebook focuses on cleaning and preprocessing the raw manufacturing datasets to improve data quality before performing exploratory data analysis, SQL implementation, statistical analysis, machine learning, and dashboard development.

## Objectives

The main objectives of this notebook are to:

- Assess the quality of each dataset.
- Identify missing values, duplicate records, and inconsistent data.
- Correct data types where required.
- Standardize categorical values.
- Detect and handle outliers.
- Validate business rules across related tables.
- Prepare clean datasets for downstream analysis.

In [1]:
# Import required libraries
import pandas as pd 
import numpy as np 

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
data_path = "../📁Data/📁Raw_data/"

In [3]:
factories = pd.read_csv(data_path + "factories.csv")
production_lines = pd.read_csv(data_path + "production_lines.csv")
machines = pd.read_csv(data_path + "machines.csv")
suppliers = pd.read_csv(data_path + "suppliers.csv")
products = pd.read_csv(data_path + "products.csv")
operators = pd.read_csv(data_path + "operators.csv")
production_batches = pd.read_csv(data_path + "production_batches.csv")
sensor_readings = pd.read_csv(data_path + "sensor_readings.csv")
maintenance_work_orders = pd.read_csv(data_path + "maintenance_work_orders.csv")
quality_inspections = pd.read_csv(data_path + "quality_inspections.csv")
downtime_events = pd.read_csv(data_path + "downtime_events.csv")
failure_events = pd.read_csv(data_path + "failure_events.csv")

In [4]:
datasets = {
    "Factories": factories,
    "Production Lines": production_lines,
    "Machines": machines,
    "Suppliers": suppliers,
    "Products": products,
    "Operators": operators,
    "Production Batches": production_batches,
    "Sensor Readings": sensor_readings,
    "Maintenance Work Orders": maintenance_work_orders,
    "Quality Inspections": quality_inspections,
    "Downtime Events": downtime_events,
    "Failure Events": failure_events,
}

## Data Quality Assessment

Before performing any cleaning operations, it is important to assess the overall quality of the datasets. This assessment helps identify potential issues such as missing values, duplicate records, incorrect data types, and memory usage, allowing us to plan an appropriate cleaning strategy.

In [5]:
quality_check = pd.DataFrame({
    "Dataset" : datasets.keys(),
    "Rows" : [df.shape[0] for df in datasets.values()],
    "Columns" : [df.shape[1] for df in datasets.values()],
    "Missing Values" : [df.isnull().sum().sum() for df in datasets.values()],
    "Duplicate Rows" : [df.duplicated().sum() for df in datasets.values()],
    "Memory usuage (MB)":[
        round(df.memory_usage(deep=True).sum()/ 1024**2,2)
        for df in datasets.values()],
})

quality_check

,Dataset,Rows,Columns,Missing Values,Duplicate Rows,Memory usuage (MB)
0,Factories,10,9,0,0,0.00
1,Production Lines,30,7,0,0,0.01
2,Machines,600,17,0,0,0.33
3,Suppliers,210,8,0,0,0.07
4,Products,150,10,0,0,0.06
5,Operators,1000,9,0,0,0.35
6,Production Batches,60000,24,450,0,41.54
7,Sensor Readings,285000,17,278485,0,125.22
8,Maintenance Work Orders,12000,16,9000,0,8.34
9,Quality Inspections,30000,18,192,0,22.11


In [6]:
maintenance_work_orders.isnull().sum().sort_values(ascending=False)

linked_failure_event_id       9000
work_order_id                    0
line_id                          0
machine_id                       0
factory_id                       0
scheduled_timestamp              0
completion_timestamp             0
maintenance_type                 0
maintenance_reason               0
action_taken                     0
maintenance_downtime_hours       0
parts_cost_inr                   0
labor_cost_inr                   0
total_maintenance_cost_inr       0
technician_id                    0
work_order_status                0
dtype: int64

In [7]:
maintenance_work_orders[
    maintenance_work_orders["linked_failure_event_id"].isnull()
]["maintenance_type"].value_counts()

maintenance_type
Preventive    4740
Predictive    3015
Corrective    1245
Name: count, dtype: int64

In [8]:
maintenance_work_orders[
    (maintenance_work_orders["maintenance_type"] == "Corrective") &
    (maintenance_work_orders["linked_failure_event_id"].isnull())
]

,work_order_id,machine_id,line_id,factory_id,linked_failure_event_id,scheduled_timestamp,completion_timestamp,maintenance_type,maintenance_reason,action_taken,maintenance_downtime_hours,parts_cost_inr,labor_cost_inr,total_maintenance_cost_inr,technician_id,work_order_status
3003,MWO003004,MCH0276,LIN012,FAC004,NaN,2023-07-14 02:55:58,2023-07-14 07:39:10,Corrective,Calibration Due,Alignment,4.72,19314.26,10696.78,30011.04,TECH113,Completed
3004,MWO003005,MCH0415,LIN008,FAC003,NaN,2026-01-09 08:47:24,2026-01-09 19:18:00,Corrective,Calibration Due,Alignment,10.51,11064.14,10182.30,21246.44,TECH059,Completed
3005,MWO003006,MCH0463,LIN018,FAC006,NaN,2023-07-25 22:46:52,2023-07-26 11:40:52,Corrective,Rising Temperature,Alignment,12.90,26567.07,22664.60,49231.67,TECH140,Completed
3009,MWO003010,MCH0155,LIN022,FAC008,NaN,2023-10-08 12:13:30,2023-10-08 13:38:42,Corrective,Rising Temperature,Electrical Repair,1.42,15094.66,1678.97,16773.63,TECH138,Completed
3014,MWO003015,MCH0061,LIN026,FAC009,NaN,2024-07-26 07:54:42,2024-07-26 14:29:30,Corrective,Calibration Due,Inspection,6.58,10886.29,11586.32,22472.61,TECH059,Completed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11964,MWO011965,MCH0073,LIN002,FAC001,NaN,2025-12-17 16:03:54,2025-12-17 17:00:54,Corrective,Rising Temperature,Replace Component,0.95,18232.72,1440.29,19673.01,TECH046,Completed
11965,MWO011966,MCH0525,LIN017,FAC006,NaN,2026-05-03 15:46:05,2026-05-03 19:55:41,Corrective,Oil Degradation,Inspection,4.16,11067.88,6868.40,17936.28,TECH137,Completed
11992,MWO011993,MCH0129,LIN003,FAC001,NaN,2025-01-13 10:30:56,2025-01-13 16:12:20,Corrective,High Vibration,Alignment,5.69,17463.23,6385.21,23848.44,TECH061,Completed
11996,MWO011997,MCH0567,LIN016,FAC006,NaN,2023-07-10 11:59:29,2023-07-10 17:03:41,Corrective,Rising Temperature,Replace Component,5.07,24174.95,6525.36,30700.31,TECH015,Completed


In [9]:
maintenance_work_orders[
    maintenance_work_orders["linked_failure_event_id"].isnull()
]["maintenance_reason"].value_counts()

maintenance_reason
Rising Temperature    1551
High Vibration        1547
Calibration Due       1490
Scheduled Service     1476
Tool Wear             1472
Oil Degradation       1464
Name: count, dtype: int64

### Observation

The `linked_failure_event_id` column contains missing values in maintenance records. A detailed review shows that these records are associated with maintenance reasons such as **Rising Temperature**, **High Vibration**, **Calibration Due**, **Scheduled Service**, **Tool Wear**, and **Oil Degradation**.

These reasons represent preventive, predictive, or condition-based maintenance activities where no actual machine failure occurred. Since there is no corresponding failure event, the absence of a `linked_failure_event_id` is expected.

**Decision:** The missing values are considered business-valid and will be retained without modification.

In [10]:
sensor_readings.isnull().sum()

sensor_reading_id               0
machine_id                      0
line_id                         0
factory_id                      0
reading_timestamp               0
temperature_c                1200
vibration_mm_s                  0
pressure_bar                    0
oil_level_percent               0
voltage_v                       0
current_amp                     0
rpm                             0
sound_level_db                  0
machine_health_score            0
alert_status                    0
failure_within_7_days           0
linked_failure_event_id    277285
dtype: int64

In [11]:
sensor_readings["failure_within_7_days"].value_counts()

failure_within_7_days
0    277285
1      7715
Name: count, dtype: int64

In [12]:
sensor_readings[
    sensor_readings["failure_within_7_days"] == 1
]["linked_failure_event_id"].isnull().sum()

np.int64(0)

### Observation

The `linked_failure_event_id` column contains 277,285 missing values. Further validation confirmed that all these records correspond to observations where `failure_within_7_days = 0`, indicating that no machine failure occurred within the next seven days.

Additionally, all records where `failure_within_7_days = 1` contain a valid `linked_failure_event_id`, confirming the consistency of the relationship between sensor readings and failure events.

**Decision:** The missing values are business-valid and will be retained without modification.

In [13]:
sensor_readings[
    sensor_readings["temperature_c"].isnull()
]["alert_status"].value_counts()

alert_status
Normal      1187
Warning       10
Critical       3
Name: count, dtype: int64

In [14]:
sensor_readings[
    sensor_readings["temperature_c"].isnull()
]["machine_id"].value_counts().head(10)

machine_id
MCH0499    7
MCH0545    7
MCH0274    6
MCH0156    6
MCH0031    6
MCH0189    6
MCH0300    6
MCH0044    6
MCH0184    6
MCH0209    6
Name: count, dtype: int64

### Observation

The `temperature_c` column contains **1,200** missing values (approximately **0.42%** of all sensor records).

Further investigation showed that:

- 1,177 missing values occurred when `failure_within_7_days = 0`.
- 1,187 missing values were recorded while the machine status was **Normal**.
- Missing values were distributed across many machines, with no single machine contributing a significant number of missing records.

These findings indicate that the missing values are most likely caused by occasional sensor communication or data logging issues rather than machine failures.

**Decision:** Since the proportion of missing values is very small (0.42%), the missing temperatures will be imputed using the **median temperature for each machine**, preserving machine-specific operating characteristics while minimizing the impact on the dataset.

In [15]:
production_batches.isnull().sum()

batch_id                       0
machine_id                     0
line_id                        0
factory_id                     0
product_id                     0
supplier_id                    0
operator_id                  450
production_date                0
start_time                     0
shift                          0
planned_quantity               0
units_produced                 0
good_units                     0
defective_units                0
scrap_units                    0
rework_units                   0
runtime_hours                  0
downtime_hours                 0
availability_ratio             0
actual_cycle_time_minutes      0
energy_consumption_kwh         0
scrap_cost_inr                 0
batch_status                   0
failure_near_batch             0
dtype: int64

In [16]:
production_batches[
production_batches["operator_id"].isnull()].head(10)

,batch_id,machine_id,line_id,factory_id,product_id,supplier_id,operator_id,production_date,start_time,shift,planned_quantity,units_produced,good_units,defective_units,scrap_units,rework_units,runtime_hours,downtime_hours,availability_ratio,actual_cycle_time_minutes,energy_consumption_kwh,scrap_cost_inr,batch_status,failure_near_batch
48,BAT000049,MCH0506,LIN001,FAC001,PRD0007,SUP0078,NaN,2025-08-10,01:12:43,Morning,2013,1435,1352,83,50,33,5.41,1.72,0.76,0.23,377.95,92394.22,Below Plan,0
644,BAT000645,MCH0581,LIN004,FAC002,PRD0016,SUP0177,NaN,2024-03-01,10:48:39,Morning,1285,1098,1079,19,12,7,5.02,0.87,0.85,0.27,826.51,48390.70,Below Plan,0
654,BAT000655,MCH0555,LIN021,FAC007,PRD0051,SUP0137,NaN,2023-07-11,22:46:51,Evening,1180,1116,1067,49,31,18,7.86,0.71,0.92,0.42,1657.81,18193.81,Completed,0
739,BAT000740,MCH0467,LIN027,FAC009,PRD0138,SUP0097,NaN,2026-05-16,15:46:15,Morning,1518,713,649,64,47,17,4.78,5.01,0.49,0.40,901.81,138838.97,Below Plan,1
1104,BAT001105,MCH0456,LIN003,FAC001,PRD0089,SUP0132,NaN,2024-09-22,11:08:32,Evening,1124,1070,985,85,37,48,9.12,1.14,0.89,0.51,2106.37,194370.71,Completed,0
1448,BAT001449,MCH0474,LIN003,FAC001,PRD0055,SUP0155,NaN,2024-12-06,05:24:20,Night,2269,1793,1650,143,91,52,7.05,2.00,0.78,0.24,457.65,539439.63,Below Plan,0
1597,BAT001598,MCH0362,LIN007,FAC003,PRD0083,SUP0191,NaN,2026-02-15,16:01:12,Morning,802,684,644,40,29,11,8.42,0.67,0.93,0.74,527.82,67945.25,Below Plan,0
1640,BAT001641,MCH0230,LIN017,FAC006,PRD0046,SUP0113,NaN,2024-03-08,13:59:36,Evening,2403,2018,1959,59,22,37,4.33,1.07,0.80,0.13,546.82,20729.50,Below Plan,0
1985,BAT001986,MCH0322,LIN010,FAC004,PRD0012,SUP0191,NaN,2025-04-08,07:01:30,Evening,367,340,318,22,10,12,5.14,0.52,0.91,0.91,1002.01,25270.76,Completed,0
2217,BAT002218,MCH0281,LIN027,FAC009,PRD0059,SUP0079,NaN,2025-07-31,21:16:36,Evening,1854,1526,1426,100,62,38,5.56,0.76,0.88,0.22,527.98,182289.99,Below Plan,0


In [17]:
production_batches[
    production_batches["operator_id"].isnull()
]["factory_id"].value_counts()

factory_id
FAC007    52
FAC008    52
FAC002    51
FAC009    45
FAC001    44
FAC010    43
FAC003    42
FAC006    42
FAC004    40
FAC005    39
Name: count, dtype: int64

In [18]:
production_batches[
    production_batches["operator_id"].isnull()
]["machine_id"].value_counts().head(10)

machine_id
MCH0084    5
MCH0506    4
MCH0281    4
MCH0480    4
MCH0004    4
MCH0414    4
MCH0009    4
MCH0142    4
MCH0581    3
MCH0555    3
Name: count, dtype: int64

### Observation

The `operator_id` column contains **450** missing values (0.75% of all production batches).

Further investigation showed that the missing values are distributed across:

- All production shifts.
- All factories.
- Multiple production machines.
- Both completed and below-plan batches.

No systematic pattern was identified, indicating that the missing values are most likely due to occasional data entry or recording omissions rather than a business process.

**Decision:** Since `operator_id` is an identifier and cannot be reliably inferred, the missing values will be retained without imputation.

In [19]:
downtime_events.isnull().sum()

downtime_event_id             0
machine_id                    0
line_id                       0
factory_id                    0
linked_failure_event_id    3704
linked_work_order_id          0
event_timestamp               0
downtime_cause                0
duration_hours                0
estimated_lost_units          0
estimated_loss_inr            0
resolution_status             0
impact_level                  0
dtype: int64

In [20]:
downtime_events[
    downtime_events["linked_failure_event_id"].isnull()
].head(10)

,downtime_event_id,machine_id,line_id,factory_id,linked_failure_event_id,linked_work_order_id,event_timestamp,downtime_cause,duration_hours,estimated_lost_units,estimated_loss_inr,resolution_status,impact_level
3000,DTE003001,MCH0447,LIN011,FAC004,NaN,MWO011106,2024-05-06 22:52:35,Changeover,5.57,633,432707.32,Resolved,Low
3001,DTE003002,MCH0396,LIN028,FAC010,NaN,MWO006099,2026-03-17 21:58:50,Material Shortage,0.96,196,193509.71,Resolved,High
3002,DTE003003,MCH0149,LIN004,FAC002,NaN,MWO011988,2023-02-21 03:32:46,Changeover,6.31,752,805516.90,Resolved,Medium
3003,DTE003004,MCH0474,LIN003,FAC001,NaN,MWO003229,2026-04-13 17:21:36,Tool Change,6.41,1345,209795.00,Resolved,Medium
3004,DTE003005,MCH0220,LIN023,FAC008,NaN,MWO011352,2023-05-14 04:27:20,Changeover,1.12,65,23413.86,Resolved,High
3005,DTE003006,MCH0051,LIN011,FAC004,NaN,MWO005138,2025-03-18 04:02:20,Quality Hold,3.49,459,411105.20,Resolved,High
3006,DTE003007,MCH0120,LIN007,FAC003,NaN,MWO004899,2023-06-21 02:17:32,Tool Change,4.04,517,81113.86,Resolved,High
3008,DTE003009,MCH0346,LIN015,FAC005,NaN,MWO009803,2024-07-26 11:19:47,Material Shortage,3.48,510,537842.67,Resolved,High
3009,DTE003010,MCH0599,LIN003,FAC001,NaN,MWO008070,2025-07-23 14:10:32,Material Shortage,2.48,503,339205.39,Resolved,Medium
3010,DTE003011,MCH0394,LIN001,FAC001,NaN,MWO007894,2023-09-13 07:38:55,Tool Change,2.52,538,579163.53,Resolved,High


In [21]:
[downtime_events[
    downtime_events["linked_failure_event_id"].isnull()
]["downtime_cause"].value_counts()
]

[downtime_cause
 Tool Change            645
 Quality Hold           643
 Material Shortage      635
 Planned Maintenance    634
 Power Failure          606
 Changeover             541
 Name: count, dtype: int64]

In [22]:
downtime_events[
    downtime_events["linked_failure_event_id"].notna()
]["downtime_cause"].value_counts()

downtime_cause
Machine Breakdown      3000
Changeover              233
Quality Hold            220
Material Shortage       220
Power Failure           215
Planned Maintenance     206
Tool Change             202
Name: count, dtype: int64

### Observation

The `linked_failure_event_id` column contains missing values in the downtime events dataset.

Further investigation revealed two valid business scenarios:

- Downtime events such as **Material Shortage**, **Tool Change**, **Quality Hold**, **Power Failure**, **Changeover**, and **Planned Maintenance** may occur independently of a machine failure. These records correctly contain no `linked_failure_event_id`.

- Downtime events associated with **Machine Breakdown** and some operational activities performed during failure recovery contain a valid `linked_failure_event_id`.

This indicates that the column is used to identify whether a downtime event is related to a recorded machine failure rather than describing the downtime cause itself.

**Decision:** The missing values are considered business-valid and will be retained without modification.

In [23]:
production_batches["shift"].unique()

<StringArray>
['Morning', 'Evening', 'Night', 'EVENING', 'Night Shift', 'morning']
Length: 6, dtype: str

In [24]:
production_batches["shift"] = production_batches["shift"].str.strip().str.title().replace({"Night Shift":"Night"})

In [25]:
production_batches["shift"].unique()

<StringArray>
['Morning', 'Evening', 'Night']
Length: 3, dtype: str

In [26]:
maintenance_work_orders["maintenance_type"].unique()

<StringArray>
['Corrective', 'Emergency', 'Preventive', 'Predictive']
Length: 4, dtype: str

In [27]:
downtime_events["downtime_cause"].unique()

<StringArray>
[  'Machine Breakdown',          'Changeover',   'Material Shortage',
         'Tool Change',        'Quality Hold',       'Power Failure',
 'Planned Maintenance']
Length: 7, dtype: str

In [28]:
quality_inspections["inspection_result"].unique()

<StringArray>
['Fail', 'Pass']
Length: 2, dtype: str

In [29]:
def standardize_text(series):
    return (
        series
        .str.strip()
        .str.title()
    )

In [30]:
for name, df in datasets.items():
    print("=" * 70)
    print(name)
    display(df.dtypes)

Factories


factory_id            str
factory_name          str
city                  str
state                 str
region                str
specialization        str
employee_count      int64
target_oee        float64
certification         str
dtype: object

Production Lines


line_id                   str
factory_id                str
line_name                 str
line_type                 str
daily_capacity_units    int64
shift_pattern             str
line_status               str
dtype: object

Machines


...

inspection_id                     str
batch_id                          str
machine_id                        str
line_id                           str
factory_id                        str
product_id                        str
supplier_id                       str
operator_id                       str
inspection_date                   str
inspector_id                      str
sample_size                     int64
passed_samples                  int64
failed_samples                  int64
defect_type                       str
defect_severity                   str
measured_dimension_mm         float64
inspection_result                 str
estimated_quality_loss_inr    float64
dtype: object

Downtime Events


downtime_event_id              str
machine_id                     str
line_id                        str
factory_id                     str
linked_failure_event_id        str
linked_work_order_id           str
event_timestamp                str
downtime_cause                 str
duration_hours             float64
estimated_lost_units         int64
estimated_loss_inr         float64
resolution_status              str
impact_level                   str
dtype: object

Failure Events


failure_event_id                  str
machine_id                        str
line_id                           str
factory_id                        str
failure_timestamp                 str
failure_mode                      str
failure_severity                  str
sensor_precursor_days           int64
repair_duration_hours         float64
estimated_failure_loss_inr    float64
resolution_status                 str
dtype: object

In [31]:
machines["installation_date"] = pd.to_datetime(machines["installation_date"])

In [32]:
quality_inspections["inspection_date"] = pd.to_datetime(quality_inspections["inspection_date"])

In [33]:
production_batches["production_date"] = pd.to_datetime(production_batches["production_date"])

In [34]:

production_batches["start_time"] = pd.to_datetime(
    production_batches["start_time"]
)


sensor_readings["reading_timestamp"] = pd.to_datetime(
    sensor_readings["reading_timestamp"]
)

maintenance_work_orders["scheduled_timestamp"] = pd.to_datetime(
    maintenance_work_orders["scheduled_timestamp"]
)

maintenance_work_orders["completion_timestamp"] = pd.to_datetime(
    maintenance_work_orders["completion_timestamp"]
)

downtime_events["downtime_start"] = pd.to_datetime(
    downtime_events["event_timestamp"]
)


failure_events["failure_timestamp"] = pd.to_datetime(
    failure_events["failure_timestamp"])

C:\Users\gunnu\AppData\Local\Temp\ipykernel_30084\3643843763.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  production_batches["start_time"] = pd.to_datetime(


In [35]:
sensor_readings.describe()

,reading_timestamp,temperature_c,vibration_mm_s,pressure_bar,oil_level_percent,voltage_v,current_amp,rpm,sound_level_db,machine_health_score,failure_within_7_days
count,285000,283800.00,285000.00,285000.00,285000.00,285000.00,285000.00,285000.00,285000.00,285000.00,285000.00
mean,2024-10-01 04:56:28.203803,101.21,3.23,6.03,70.06,415.66,41.91,1366.30,72.49,84.63,0.03
min,2023-01-01 00:07:35,15.51,-1.32,2.66,20.03,366.03,13.50,0.00,53.52,5.00,0.00
25%,2023-11-16 23:24:34.750000,51.32,2.41,5.54,65.19,408.48,37.46,1003.00,69.52,80.00,0.00
50%,2024-09-29 20:27:59,63.93,3.15,6.02,70.30,415.27,41.75,1585.20,72.35,85.00,0.00
75%,2025-08-16 12:00:27.750000,74.09,3.89,6.50,75.36,422.15,46.11,1902.30,75.24,91.00,0.00
max,2026-06-30 23:50:49,450.58,59.72,9.87,104.90,1096.81,86.04,2823.60,101.02,98.00,1.00
std,NaN,103.47,1.72,0.73,8.01,16.11,6.67,686.23,4.48,8.54,0.16


In [37]:
failure_events.describe()

,failure_timestamp,sensor_precursor_days,repair_duration_hours,estimated_failure_loss_inr
count,3000,3000.00,3000.00,3000.00
mean,2024-09-14 06:12:40.424000,4.04,8.27,592998.39
min,2023-01-01 20:12:10,1.00,0.12,4479.10
25%,2023-10-21 13:43:59,2.00,3.88,232604.71
50%,2024-09-06 00:39:26.500000,4.00,7.08,455114.39
75%,2025-07-28 07:11:26,6.00,11.57,815232.14
max,2026-06-30 22:38:57,7.00,38.85,3651388.38
std,NaN,1.98,5.53,485063.27


In [38]:
machines.describe()

,installation_date,machine_age_years,rated_capacity_units_per_hour,normal_temperature_c,warning_temperature_c,normal_rpm_min,normal_rpm_max,baseline_vibration_mm_s,base_failure_probability
count,600,600.00,600.00,600.00,600.00,600.00,600.00,600.00,600.00
mean,2017-07-24 05:04:48,8.93,289.92,99.39,139.76,1081.67,1642.50,2.56,0.05
min,2010-01-07 00:00:00,1.50,80.00,40.00,65.00,0.00,0.00,1.50,0.02
25%,2014-01-13 06:00:00,5.30,186.00,45.00,70.00,700.00,1300.00,2.00,0.04
50%,2017-06-21 00:00:00,9.05,290.00,65.00,80.00,1200.00,1900.00,2.70,0.05
75%,2021-03-11 00:00:00,12.50,400.00,72.00,85.00,1500.00,2200.00,3.00,0.06
max,2024-12-14 00:00:00,16.50,500.00,350.00,520.00,1800.00,2400.00,3.40,0.09
std,NaN,4.29,122.40,103.13,155.79,572.34,775.68,0.63,0.02


In [39]:
maintenance_work_orders.describe()

,scheduled_timestamp,completion_timestamp,maintenance_downtime_hours,parts_cost_inr,labor_cost_inr,total_maintenance_cost_inr
count,12000,12000,12000.00,12000.00,12000.00,12000.00
mean,2024-09-30 22:43:51.800833,2024-10-01 03:24:45.923833,4.68,19556.99,7820.70,27377.69
min,2023-01-01 04:56:23,2023-01-01 07:06:37,0.05,148.52,73.25,1401.53
25%,2023-11-09 16:39:40,2023-11-09 21:39:37,2.20,8919.55,3427.19,15664.72
50%,2024-10-02 23:38:14.500000,2024-10-03 02:48:59,3.91,15889.65,6322.41,23711.11
75%,2025-08-17 07:46:20,2025-08-17 11:06:17.250000,6.37,26505.56,10554.85,34872.86
max,2026-06-30 23:35:39,2026-07-01 02:48:15,28.65,111599.65,54955.47,123499.38
std,NaN,NaN,3.34,14411.53,5997.66,16327.63


In [41]:
production_batches.describe()

,production_date,start_time,planned_quantity,units_produced,good_units,defective_units,scrap_units,rework_units,runtime_hours,downtime_hours,availability_ratio,actual_cycle_time_minutes,energy_consumption_kwh,scrap_cost_inr,failure_near_batch
count,60000,60000,60000.00,60000.00,60000.00,60000.00,60000.00,60000.00,60000.00,60000.00,60000.00,60000.00,60000.00,60000.00,60000.00
mean,2024-09-30 16:13:32.160000,2026-07-23 12:01:09.547283,1652.59,1451.62,1342.02,109.60,59.75,49.85,8.00,1.11,0.88,0.47,1218.71,151831.44,0.03
min,2023-01-01 00:00:00,2026-07-23 00:00:00,300.00,136.00,115.00,0.00,0.00,0.00,4.00,0.10,0.31,0.09,208.31,0.00,0.00
25%,2023-11-19 00:00:00,2026-07-23 06:03:07,974.75,846.00,781.00,57.00,29.00,24.00,6.02,0.63,0.85,0.23,725.29,50345.02,0.00
50%,2024-09-30 00:00:00,2026-07-23 12:01:53.500000,1651.00,1433.00,1322.00,99.00,52.00,43.00,8.00,0.89,0.90,0.33,1069.07,110898.46,0.00
75%,2025-08-12 00:00:00,2026-07-23 17:58:35,2334.00,2041.00,1888.00,152.00,82.00,68.00,9.99,1.25,0.93,0.57,1537.30,211559.22,0.00
max,2026-06-30 00:00:00,2026-07-23 23:59:58,3000.00,3128.00,2976.00,452.00,309.00,241.00,12.00,10.30,0.99,3.60,25880.00,1524431.32,1.00
std,NaN,NaN,781.43,705.35,654.37,65.01,38.58,32.79,2.30,1.00,0.08,0.38,1146.16,137937.95,0.16


In [42]:
production_lines.describe()

,daily_capacity_units
count,30.00
mean,2403.13
std,808.76
min,537.00
25%,1981.00
50%,2425.00
75%,2934.25
max,3972.00


In [43]:
products.describe()

,target_cycle_time_minutes,standard_unit_cost_inr,dimensional_tolerance_mm
count,150.00,150.00,150.00
mean,4.83,3395.07,0.24
std,2.18,1785.49,0.14
min,0.72,121.22,0.01
25%,2.79,1998.81,0.13
50%,4.58,3357.53,0.24
75%,6.82,4996.94,0.35
max,8.43,6468.55,0.49


In [44]:
quality_inspections.describe()

,inspection_date,sample_size,passed_samples,failed_samples,measured_dimension_mm,estimated_quality_loss_inr
count,30000,30000.00,30000.00,30000.00,30000.00,30000.00
mean,2024-10-01 15:00:54.720000,139.93,126.66,13.27,50.00,17323.91
min,2023-01-01 00:00:00,30.00,20.00,0.00,46.91,0.00
25%,2023-11-22 00:00:00,85.00,76.00,7.00,49.55,4007.12
50%,2024-09-27 00:00:00,140.00,126.00,12.00,50.01,10302.32
75%,2025-08-14 00:00:00,196.00,177.00,18.00,50.46,22665.31
max,2026-06-30 00:00:00,250.00,246.00,64.00,53.30,329190.51
std,NaN,63.92,58.20,8.40,0.75,20837.53


In [45]:
suppliers.describe()

,quality_rating,average_lead_time_days,delivery_reliability
count,210.00,210.00,210.00
mean,0.91,19.85,0.88
std,0.05,8.92,0.06
min,0.82,3.00,0.78
25%,0.87,13.00,0.82
50%,0.90,19.00,0.88
75%,0.95,27.00,0.93
max,0.99,35.00,0.99


In [46]:
downtime_events.describe()

,duration_hours,estimated_lost_units,estimated_loss_inr,downtime_start
count,8000.00,8000.00,8000.00,8000
mean,5.26,733.76,497824.12,2024-09-24 14:25:08.521625
min,0.04,3.00,1759.68,2023-01-01 06:25:13
25%,2.02,224.00,123467.73,2023-11-06 17:31:28.250000
50%,3.78,480.50,283682.55,2024-09-24 23:25:32
75%,7.08,976.00,621218.37,2025-08-06 22:14:26.500000
max,38.85,8135.00,9508950.61,2026-06-30 23:35:39
std,4.63,757.18,620876.34,NaN


In [49]:
(sensor_readings["temperature_c"]>200).sum()

np.int64(40555)

In [50]:
sensor_machine = sensor_readings.merge(
    machines[["machine_id", "warning_temperature_c"]],
    on="machine_id",
    how="left"
)

(sensor_machine["temperature_c"] >
 sensor_machine["warning_temperature_c"]).sum()

np.int64(5306)

In [51]:
sensor_readings[sensor_readings["vibration_mm_s"] < 0]

,sensor_reading_id,machine_id,line_id,factory_id,reading_timestamp,temperature_c,vibration_mm_s,pressure_bar,oil_level_percent,voltage_v,current_amp,rpm,sound_level_db,machine_health_score,alert_status,failure_within_7_days,linked_failure_event_id
343,SNS0000344,MCH0043,LIN023,FAC008,2024-02-02 21:02:11,297.30,-0.23,6.03,81.15,406.32,34.08,0.00,69.19,86,Normal,0,NaN
964,SNS0000965,MCH0180,LIN011,FAC004,2026-05-24 11:52:44,348.87,-0.59,5.07,73.36,416.36,33.65,0.00,67.82,97,Normal,0,NaN
1368,SNS0001369,MCH0127,LIN004,FAC002,2025-07-06 10:10:46,50.93,-0.40,5.61,81.03,411.08,23.22,1260.50,70.67,95,Normal,0,NaN
1410,SNS0001411,MCH0450,LIN022,FAC008,2025-06-06 00:44:59,47.19,-0.21,5.11,68.66,411.94,36.55,809.20,68.87,96,Normal,0,NaN
3166,SNS0003167,MCH0291,LIN022,FAC008,2025-09-02 22:35:44,355.17,-0.37,7.22,64.52,419.73,57.88,0.00,71.49,89,Normal,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282615,SNS0282616,MCH0085,LIN020,FAC007,2024-03-15 13:31:39,369.75,-0.06,7.18,80.22,425.68,37.17,0.00,72.08,93,Normal,0,NaN
284163,SNS0284164,MCH0394,LIN001,FAC001,2024-06-11 06:37:45,343.96,-0.17,6.17,75.13,403.76,39.21,0.00,68.10,96,Normal,0,NaN
284185,SNS0284186,MCH0396,LIN028,FAC010,2025-02-25 03:09:33,372.01,-0.45,5.89,55.37,416.97,40.58,0.00,75.97,89,Normal,0,NaN
284779,SNS0284780,MCH0351,LIN021,FAC007,2024-05-21 03:36:06,309.29,-0.22,7.33,63.87,408.91,55.86,0.00,76.35,92,Normal,0,NaN


In [58]:
sensor_readings.loc[
    sensor_readings["vibration_mm_s"] <= 0,
    "vibration_mm_s"
] = np.nan

In [62]:
sensor_readings[
    sensor_readings["oil_level_percent"] > 100
]

,sensor_reading_id,machine_id,line_id,factory_id,reading_timestamp,temperature_c,vibration_mm_s,pressure_bar,oil_level_percent,voltage_v,current_amp,rpm,sound_level_db,machine_health_score,alert_status,failure_within_7_days,linked_failure_event_id
28435,SNS0028436,MCH0107,LIN010,FAC004,2023-07-10 15:30:54,298.39,1.31,5.54,101.32,412.12,37.69,0.00,73.73,91,Normal,0,NaN
45793,SNS0045794,MCH0348,LIN017,FAC006,2025-05-12 21:59:58,67.78,4.15,5.53,100.07,416.14,42.33,1783.00,61.46,93,Normal,0,NaN
62052,SNS0062053,MCH0236,LIN021,FAC007,2023-03-09 04:20:32,387.97,1.23,5.60,100.27,405.72,35.33,0.00,74.30,97,Normal,0,NaN
109398,SNS0109399,MCH0316,LIN029,FAC010,2023-09-22 17:36:59,46.07,0.71,6.07,104.90,421.25,36.11,1800.00,65.98,93,Normal,0,NaN
134995,SNS0134996,MCH0552,LIN012,FAC004,2024-05-30 20:24:12,57.30,3.99,6.52,100.52,403.73,45.57,1342.10,73.50,83,Normal,0,NaN
148841,SNS0148842,MCH0506,LIN001,FAC001,2023-02-24 08:36:13,67.86,2.07,6.13,101.13,410.48,34.72,1843.50,67.86,92,Normal,0,NaN
215283,SNS0215284,MCH0491,LIN021,FAC007,2024-08-31 02:15:21,76.28,3.49,5.11,100.97,432.21,39.92,2115.00,73.69,92,Normal,0,NaN


In [63]:
sensor_readings[
    sensor_readings["oil_level_percent"] > 100
][
    [
        "sensor_reading_id",
        "machine_id",
        "reading_timestamp",
        "oil_level_percent",
        "failure_within_7_days",
        "alert_status"
    ]
]

,sensor_reading_id,machine_id,reading_timestamp,oil_level_percent,failure_within_7_days,alert_status
28435,SNS0028436,MCH0107,2023-07-10 15:30:54,101.32,0,Normal
45793,SNS0045794,MCH0348,2025-05-12 21:59:58,100.07,0,Normal
62052,SNS0062053,MCH0236,2023-03-09 04:20:32,100.27,0,Normal
109398,SNS0109399,MCH0316,2023-09-22 17:36:59,104.90,0,Normal
134995,SNS0134996,MCH0552,2024-05-30 20:24:12,100.52,0,Normal
148841,SNS0148842,MCH0506,2023-02-24 08:36:13,101.13,0,Normal
215283,SNS0215284,MCH0491,2024-08-31 02:15:21,100.97,0,Normal


In [64]:
sensor_readings.loc[
    sensor_readings["oil_level_percent"] > 100,
    "oil_level_percent"
] = 100

In [65]:
sensor_readings.describe()

,reading_timestamp,temperature_c,vibration_mm_s,pressure_bar,oil_level_percent,voltage_v,current_amp,rpm,sound_level_db,machine_health_score,failure_within_7_days
count,285000,283800.00,284656.00,285000.00,285000.00,285000.00,285000.00,285000.00,285000.00,285000.00,285000.00
mean,2024-10-01 04:56:28.203803,101.21,3.23,6.03,70.06,415.66,41.91,1366.30,72.49,84.63,0.03
min,2023-01-01 00:07:35,15.51,0.00,2.66,20.03,366.03,13.50,0.00,53.52,5.00,0.00
25%,2023-11-16 23:24:34.750000,51.32,2.41,5.54,65.19,408.48,37.46,1003.00,69.52,80.00,0.00
50%,2024-09-29 20:27:59,63.93,3.15,6.02,70.30,415.27,41.75,1585.20,72.35,85.00,0.00
75%,2025-08-16 12:00:27.750000,74.09,3.89,6.50,75.36,422.15,46.11,1902.30,75.24,91.00,0.00
max,2026-06-30 23:50:49,450.58,59.72,9.87,100.00,1096.81,86.04,2823.60,101.02,98.00,1.00
std,NaN,103.47,1.71,0.73,8.01,16.11,6.67,686.23,4.48,8.54,0.16


In [66]:
import os

cleaned_path = "../📁Data/📁Cleaned_Data"
os.makedirs(cleaned_path, exist_ok=True)

for table_name, df in datasets.items():
    df.to_csv(f"{cleaned_path}/{table_name}.csv", index=False)

print("✅ All cleaned datasets have been saved successfully.")

✅ All cleaned datasets have been saved successfully.
